# Single Non-Interactive Distributed Simulation with pystclient

This example demonstrates how to run a **single non-interactive** distributed
simulation using an existing Spring-Mass-Damper (MSD) project, including
parameter configuration, execution, result collection, and plotting.


## 1 - Authenticate and select project


In [ ]:
from pystclient.clients import PyStclient

client = PyStclient()
_ = client.authenticate()

# Select an existing project by name
projects = client.project.info_all()
project_info = next(p for p in projects if p.name == "Tutorial")
print(f"Using project: {project_info.name}  (id={project_info.id})")

## 2 - Single Non-Interactive Distributed Simulation
A simulation is **non-interactive** when the simulator runs to completion
without user intervention (start -> run -> end). Here we create a single
distributed simulator with one parameter set, start it with recording, wait
for it to finish, and collect the results.


In [ ]:
from pystclient.models import (
    LoggingConfiguration,
    ModelParameters,
    ModelVariable,
    SimulationParameters,
)

# Configure simulation parameters with model-specific settings
sim_params = SimulationParameters(
    base_step_size=0.01,
    end_time=20,
    config_name="Config 1",
    model_parameters=[
        ModelParameters(
            name="Mass1",
            step_size=0.02,
            parameters=[ModelVariable(name="absTolerance", initial_value=2e-4)],
        ),
        ModelParameters(
            name="Spring1",
            parameters=[ModelVariable(name="absTolerance", initial_value=2e-4)],
        ),
        ModelParameters(
            name="Damper1",
            parameters=[ModelVariable(name="absTolerance", initial_value=2e-4)],
        ),
    ],
)
assert client.project.update_parameters(project_info.id, sim_params)

# Enable post-plotting so results are persisted
log_config = LoggingConfiguration(post_plotting=True)
assert client.project.update_log_config(project_info.id, log_config)

In [ ]:
import time

from pystclient.clients import SimulationResults
from pystclient.models import SimulationConfig
from pystclient.types import SimulationType

# Create a single non-interactive distributed simulator
statuses, future = client.simulator.create(
    SimulationConfig(
        project_id=project_info.id,
        parameter_set_names=["Config 1"],
        type=SimulationType.DISTRIBUTED,
    )
)
simulator_id = statuses[0].id
print(f"Simulator created: {simulator_id}  status={statuses[0].status}")

# Wait until the simulator is ready (node may need to scale up)
assert future.result(600), "Simulator did not become ready in time!"
print("Simulator is ready.")

# Start the simulation with recording enabled
assert client.simulator.start(simulator_id, record=True)
print("Simulation started (recording).")

# Poll until the simulation finishes
while not client.simulator.finished(simulator_id):
    s = client.simulator.status(simulator_id)
    print(f"  simulation_time={s.simulation_time}  end_time={s.end_time}")
    time.sleep(2)

print("Simulation finished.")

# End the simulation and wait for measurement results
results_future = client.simulator.end_simulation(simulator_id)
single_results: SimulationResults | None = results_future.result(120)
assert isinstance(single_results, SimulationResults), "No results returned!"
print(f"Results available - {single_results.measurement_size()} measurement(s).")

### 2.1 - Plot displacement from single simulation


In [ ]:
# pyright: reportUnknownMemberType=false
import matplotlib.pyplot as plt

from pystclient.models import QueryResult
from pystclient.utils.time import convert_to_timestamp

single_results.reset()

fig, ax = plt.subplots(figsize=(10, 5))

for query_result in single_results:
    for result in query_result:
        result: QueryResult
        if result.signal == "dis_yx":
            x = convert_to_timestamp(result.x)
            ax.plot(x, result.y, label=f"{result.module} - {result.signal}")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Displacement [m]")
ax.set_title("Single Simulation - Spring Displacement (dis_yx)")
ax.legend()
plt.tight_layout()
plt.show()